# DEEP R on multi-MNIST 2-task with LTU hidden layer (input-layer pruning)

Same multi-MNIST 2-task setting as the other notebooks in this directory, with
a 1-hidden-layer LTU model. Each hidden unit is hardwired to exactly one
output (1-to-1 routing), and each unit starts with `INPUT_FANIN` random input
connections. Hidden units are equally split across the 20 outputs.

Training:
- **Input layer (W1)**: DEEP R update — gradient + L1 decay + Gaussian noise
  on active entries; deactivate (prune) when an active weight changes sign.
- **Output layer (W2)**: same DEEP R-style update, but **no pruning**: weights
  stay active even when they cross zero. M2 stays at the initial 1-per-row
  routing throughout.
- **Baseline**: same init, plain SGD with frozen masks (no L1, no noise, no
  pruning). Used as a no-prune reference loss.

Metrics over training:
- DEEP R loss vs baseline loss.
- Cross/within input-layer pruned counts (averaged across hidden units), and
  the cross/within prune ratio.
- Cross/within fraction of initial input-layer connections that remain.

Future-proofing notes (NOT implemented now, just noted): we will later add
more outgoing connections, add new features (more inputs/units), prune
features, and run continual rewiring. The train function takes separate
W/M arrays for both layers and doesn't bake in `M2 sum-per-row == 1`, so
those extensions are additive.

## Setup

In [1]:
import os
import sys

REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from phd.jax_core.models import ltu

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS              # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS                # 20

# 2-layer LTU layout — each hidden unit hardwired to one output, equally split.
N_HIDDEN = 100                                    # must divide OUTPUT_DIM
HIDDEN_PER_OUTPUT = N_HIDDEN // OUTPUT_DIM        # 5
INPUT_FANIN = 256                                 # initial active input connections per hidden unit

assert N_HIDDEN % OUTPUT_DIM == 0, "N_HIDDEN must be divisible by OUTPUT_DIM"

print('JAX device:', jax.devices()[0])
print(f'INPUT_DIM={INPUT_DIM}  N_HIDDEN={N_HIDDEN}  OUTPUT_DIM={OUTPUT_DIM}'
      f'  HIDDEN_PER_OUTPUT={HIDDEN_PER_OUTPUT}  INPUT_FANIN={INPUT_FANIN}')

JAX device: cuda:0
INPUT_DIM=1568  N_HIDDEN=100  OUTPUT_DIM=20  HIDDEN_PER_OUTPUT=5  INPUT_FANIN=256


## Data

In [2]:
def load_data():
    """MNIST standardized per-pixel."""
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, '   labels:', labels.shape)

images: (60000, 784)    labels: (60000,)


## Architecture

Forward: `z1 = x @ (W1 * M1); h = ltu(z1); logits = h @ (W2 * M2)`. Per-task
softmax CE summed over 2 tasks. No biases.

In [3]:
def forward(W1, M1, W2, M2, x):
    z1 = x @ (W1 * M1)                              # (N_HIDDEN,)
    h = ltu(z1)                                     # binary {0,1} forward, sigmoid-STE backward
    logits = h @ (W2 * M2)                          # (OUTPUT_DIM,)
    return logits, h, z1


def loss_fn(W1, M1, W2, M2, x, y):
    logits, _, _ = forward(W1, M1, W2, M2, x)
    logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)
    lp = jax.nn.log_softmax(logits_pt, axis=-1)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, NUM_CLASSES) * lp, axis=-1))


def make_sample(images, labels, key):
    k1, k2 = jax.random.split(key)
    idx1 = jax.random.randint(k1, (), 0, images.shape[0])
    idx2 = jax.random.randint(k2, (), 0, images.shape[0])
    x = jnp.concatenate([images[idx1], images[idx2]])
    y = jnp.array([labels[idx1], labels[idx2]])
    return x, y

## Init

Per hidden unit: pick `INPUT_FANIN` random input pixels; one-hot to its assigned
output. Weights at active entries are Kaiming-uniform with the appropriate fan-in.
Future extensions (more outgoing connections, feature growth) just modify these
masks before passing them to the train functions.

In [4]:
def init_2layer_ltu(seed=0, n_hidden=N_HIDDEN, input_fanin=INPUT_FANIN):
    """Init W1, M1, W2, M2.

    M1: each column has `input_fanin` ones at random rows.
    M2: one-hot row, hidden unit i routes to output i // (n_hidden // OUTPUT_DIM).
    """
    k = jax.random.key(seed)
    k_m1, k_w1, k_w2 = jax.random.split(k, 3)

    # M1: per hidden unit, sample input_fanin random inputs.
    keys = jax.random.split(k_m1, n_hidden)

    def per_unit(key):
        noise = jax.random.uniform(key, (INPUT_DIM,))
        idx = jnp.argsort(-noise)[:input_fanin]
        return jnp.zeros(INPUT_DIM, dtype=jnp.int32).at[idx].set(1)

    M1_T = jax.vmap(per_unit)(keys)                              # (N_HIDDEN, INPUT_DIM)
    M1 = M1_T.T                                                  # (INPUT_DIM, N_HIDDEN)

    w1_bound = jnp.sqrt(3.0 / float(input_fanin))
    W1 = jax.random.uniform(k_w1, (INPUT_DIM, n_hidden),
                            minval=-w1_bound, maxval=w1_bound) * M1

    hidden_per_output = n_hidden // OUTPUT_DIM
    h2o = jnp.arange(n_hidden) // hidden_per_output              # (N_HIDDEN,)
    M2 = jax.nn.one_hot(h2o, OUTPUT_DIM, dtype=jnp.int32)        # (N_HIDDEN, OUTPUT_DIM)
    w2_bound = jnp.sqrt(3.0 / float(hidden_per_output))
    W2 = jax.random.uniform(k_w2, (n_hidden, OUTPUT_DIM),
                            minval=-w2_bound, maxval=w2_bound) * M2
    return W1, M1, W2, M2


# Sanity-check the init.
_W1, _M1, _W2, _M2 = init_2layer_ltu(seed=0)
print(f'M1 active per hidden unit: min={int(_M1.sum(0).min())} '
      f'mean={float(_M1.sum(0).mean())} max={int(_M1.sum(0).max())}')
print(f'M2 active per hidden unit: {int(_M2.sum(1).min())} (should be 1)')
print(f'M2 active per output: {int(_M2.sum(0).min())}/{int(_M2.sum(0).max())} '
      f'(should both be {HIDDEN_PER_OUTPUT})')

M1 active per hidden unit: min=256 mean=256.0 max=256
M2 active per hidden unit: 1 (should be 1)
M2 active per output: 5/5 (should both be 5)


## DEEP R training

Per step on each layer:

  W_new = W − lr·grad − lr·l1·sign(W) + sqrt(2·lr·T)·noise

apply current mask, then for **W1 only** deactivate any entry whose sign just
flipped (and was active). W2 uses the same update but keeps its mask fixed.

Set `temperature=0` for deterministic pruning; `l1=0` makes the active set
equilibrium-stable (prunes only happen when gradient pushes a weight to 0).

In [5]:
def train_deep_r(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                 lr=2**-7,
                 l1=1e-4,
                 temperature=1e-7,
                 n_steps=500_000,
                 snapshot_every=2_000,
                 permute_period=0,
                 seed=0):
    """DEEP R on W1 (with deactivation) and W2 (no deactivation).

    `permute_period`: 0 = stationary multi-MNIST. >0 = every `permute_period`
    training steps, randomly pick a task and apply a random permutation to its
    class-label mapping (non-stationary task)."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    init_carry = (W1_init, M1_init.astype(jnp.int32),
                  W2_init, M2_init.astype(jnp.int32),
                  perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))
    noise_scale = jnp.sqrt(2.0 * lr * temperature)

    def step_fn(carry, key):
        W1, M1, W2, M2, perm0, perm1, t = carry
        data_key, n1_key, n2_key, perm_key = jax.random.split(key, 4)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])

        # value_and_grad on W1 and W2 only (masks are arguments but not differentiated).
        def _loss(W1_, W2_):
            return loss_fn(W1_, M1, W2_, M2, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)

        s1 = jnp.sign(W1)
        s2 = jnp.sign(W2)
        n1 = jax.random.normal(n1_key, W1.shape) * noise_scale
        n2 = jax.random.normal(n2_key, W2.shape) * noise_scale

        # DEEP R update — apply, then mask to keep inactive at 0.
        W1_new = (W1 - lr * g1 - lr * l1 * s1 + n1) * M1
        W2_new = (W2 - lr * g2 - lr * l1 * s2 + n2) * M2

        # W1: deactivate active entries that flipped sign.
        deact = (jnp.sign(W1_new) != s1) & (M1 == 1)
        M1_new = M1 * (1 - deact.astype(jnp.int32))
        W1_new = W1_new * M1_new

        # W2: no deactivation; M2 unchanged.
        M2_new = M2

        t_next = t + 1
        # Optional task-class label permutation: every permute_period steps,
        # pick a task at random and apply a fresh permutation to its labels.
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0_new = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1_new = jnp.where(should_perm & (which == 1), new_perm, perm1)
        else:
            perm0_new = perm0
            perm1_new = perm1

        return (W1_new, M1_new, W2_new, M2_new, perm0_new, perm1_new, t_next), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W1, M1, W2, M2, _perm0, _perm1, t = carry
        snap = dict(
            step=t,
            avg_loss=losses.mean(),
            M1=M1,
            M2=M2,
            n_active_W1=M1.sum(),
            n_active_W2=M2.sum(),
        )
        return carry, snap

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_M1'] = jax.device_get(final_carry[1])
    snaps['final_W2'] = jax.device_get(final_carry[2])
    snaps['final_M2'] = jax.device_get(final_carry[3])
    return snaps

## Baseline (no-prune)

Same init, plain SGD, masks frozen at the initial values. Loss curve is the
no-prune reference.

In [6]:
def train_baseline(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                   lr=2**-7,
                   n_steps=500_000,
                   snapshot_every=2_000,
                   permute_period=0,
                   seed=0):
    """Plain SGD with frozen masks. No L1, no noise, no deactivation.

    `permute_period`: 0 = stationary. >0 = same non-stationary mechanism as
    train_deep_r so the loss curves are directly comparable."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    init_carry = (W1_init, W2_init, perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))

    def step_fn(carry, key):
        W1, W2, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])
        def _loss(W1_, W2_):
            return loss_fn(W1_, M1_init, W2_, M2_init, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)
        W1 = (W1 - lr * g1) * M1_init
        W2 = (W2 - lr * g2) * M2_init
        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)
        return (W1, W2, perm0, perm1, t_next), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W1, W2, _p0, _p1, t = carry
        return carry, dict(step=t, avg_loss=losses.mean())

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_W2'] = jax.device_get(final_carry[1])
    return snaps

## Run

The DEEP R run takes longer because of the noise updates and the per-chunk
mask snapshots. Reduce `n_steps` for fast iteration.

In [7]:
# Shared init (same seed → same starting topology and weights for both runs).
W1_init, M1_init, W2_init, M2_init = init_2layer_ltu(seed=0)

# DEEP R hyperparameters — edit freely.
DEEP_R_CONFIG = dict(
    lr=2**-7,
    l1=1e-4,
    temperature=1e-7,
    n_steps=500_000,
    snapshot_every=2_000,
    permute_period=0,            # 0 = stationary; e.g. 4000 to permute one task every 4k steps
    seed=0,
)

train_deep_r_jit = jax.jit(
    train_deep_r,
    static_argnames=('lr', 'l1', 'temperature', 'n_steps', 'snapshot_every',
                     'permute_period', 'seed'),
)
train_baseline_jit = jax.jit(
    train_baseline,
    static_argnames=('lr', 'n_steps', 'snapshot_every', 'permute_period', 'seed'),
)

print('Running DEEP R...')
deep_r_snaps = train_deep_r_jit(W1_init, M1_init, W2_init, M2_init,
                                images, labels, **DEEP_R_CONFIG)
# Attach the init masks for downstream metrics (kept outside JIT since they're inputs).
deep_r_snaps['M1_init'] = np.asarray(M1_init)
deep_r_snaps['M2_init'] = np.asarray(M2_init)
print(f'  final loss: {float(deep_r_snaps["avg_loss"][-1]):.4f}')
print(f'  W1 active: {int(deep_r_snaps["n_active_W1"][-1])} / {int(M1_init.sum())}')

print('\nRunning baseline (no-prune)...')
baseline_snaps = train_baseline_jit(W1_init, M1_init, W2_init, M2_init, images, labels,
                                     lr=DEEP_R_CONFIG['lr'],
                                     n_steps=DEEP_R_CONFIG['n_steps'],
                                     snapshot_every=DEEP_R_CONFIG['snapshot_every'],
                                     permute_period=DEEP_R_CONFIG['permute_period'],
                                     seed=DEEP_R_CONFIG['seed'])
print(f'  final loss: {float(baseline_snaps["avg_loss"][-1]):.4f}')

Running DEEP R...
  final loss: 0.5126
  W1 active: 2083 / 25600

Running baseline (no-prune)...
  final loss: 0.3127


## Metrics — cross/within prune accounting

Each hidden unit had a random initial fanin split between within-task and
cross-task input pixels. We track per-unit and average:

- `pruned_within`, `pruned_cross`: how many initial-active connections of each type
  have been deactivated, averaged across hidden units.
- `frac_remaining_within`, `frac_remaining_cross`: fraction of each type's initial-
  active connections that are still active, averaged across hidden units.

In [8]:
# Task assignments (matching common.hidden_unit_task_ids etc.).
INPUT_TASK  = np.arange(INPUT_DIM)  // INPUT_PER_TASK                       # (IN,)
HIDDEN_TASK = (np.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES     # (HIDDEN,)
SAME_TASK_IH = (INPUT_TASK[:, None] == HIDDEN_TASK[None, :])                # (IN, HIDDEN) bool


def compute_prune_metrics(snaps):
    """From M1_init and the per-snapshot M1 history, compute per-unit cross/within
    pruned counts and fraction-remaining arrays. Shapes: (n_chunks, N_HIDDEN)."""
    M1_init = np.asarray(snaps['M1_init']).astype(np.int32)
    M1s = np.asarray(snaps['M1']).astype(np.int32)              # (n_chunks, IN, HIDDEN)

    init_within = (M1_init * SAME_TASK_IH.astype(M1_init.dtype)).sum(axis=0)   # (HIDDEN,)
    init_cross  = (M1_init * (~SAME_TASK_IH).astype(M1_init.dtype)).sum(axis=0)

    same_h    = SAME_TASK_IH.astype(M1s.dtype)[None, :, :]      # (1, IN, HIDDEN)
    not_same  = (~SAME_TASK_IH).astype(M1s.dtype)[None, :, :]
    active_within = (M1s * same_h).sum(axis=1)                  # (n_chunks, HIDDEN)
    active_cross  = (M1s * not_same).sum(axis=1)

    pruned_within = init_within[None, :] - active_within
    pruned_cross  = init_cross[None, :]  - active_cross

    # Avoid div-by-zero for any hidden unit that started with 0 of a kind.
    den_w = np.maximum(init_within[None, :], 1)
    den_c = np.maximum(init_cross[None, :],  1)
    frac_remaining_within = active_within / den_w
    frac_remaining_cross  = active_cross  / den_c

    return dict(
        steps=np.asarray(snaps['step']),
        init_within_per_unit=init_within,
        init_cross_per_unit=init_cross,
        pruned_within=pruned_within,
        pruned_cross=pruned_cross,
        frac_remaining_within=frac_remaining_within,
        frac_remaining_cross=frac_remaining_cross,
    )


metrics = compute_prune_metrics(deep_r_snaps)
print(f'mean initial within per unit: {metrics["init_within_per_unit"].mean():.1f}')
print(f'mean initial cross  per unit: {metrics["init_cross_per_unit"].mean():.1f}')
print(f'final mean pruned within / cross: {metrics["pruned_within"][-1].mean():.1f}'
      f' / {metrics["pruned_cross"][-1].mean():.1f}')
print(f'final mean fraction remaining within / cross: '
      f'{metrics["frac_remaining_within"][-1].mean():.3f}'
      f' / {metrics["frac_remaining_cross"][-1].mean():.3f}')

mean initial within per unit: 127.5
mean initial cross  per unit: 128.5
final mean pruned within / cross: 106.7 / 128.5
final mean fraction remaining within / cross: 0.164 / 0.000


## Plots

In [9]:
WITHIN_COLOR = '#1f77b4'   # blue
CROSS_COLOR  = '#d62728'   # red


def plot_loss(deep_r_snaps, baseline_snaps):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.asarray(deep_r_snaps['step']),
                             y=np.asarray(deep_r_snaps['avg_loss']),
                             mode='lines', name='DEEP R'))
    fig.add_trace(go.Scatter(x=np.asarray(baseline_snaps['step']),
                             y=np.asarray(baseline_snaps['avg_loss']),
                             mode='lines', name='baseline (no prune)',
                             line=dict(dash='dot')))
    fig.update_layout(title='Loss over training',
                      xaxis_title='step', yaxis_title='mean loss over snapshot',
                      width=900, height=420)
    fig.show()
    return fig


def plot_pruned_counts(metrics):
    """Avg cross/within pruned per hidden unit over training."""
    steps = metrics['steps']
    pw = metrics['pruned_within'].mean(axis=-1)
    pc = metrics['pruned_cross'].mean(axis=-1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=pw, mode='lines', name='within-task pruned',
                             line=dict(color=WITHIN_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=pc, mode='lines', name='cross-task pruned',
                             line=dict(color=CROSS_COLOR)))
    fig.update_layout(title='Pruned input connections (averaged over hidden units)',
                      xaxis_title='step',
                      yaxis_title='# pruned per hidden unit',
                      width=900, height=420)
    fig.show()
    return fig


def plot_pruned_ratio(metrics):
    """Ratio = mean(cross pruned) / mean(within pruned)."""
    steps = metrics['steps']
    pw = metrics['pruned_within'].mean(axis=-1)
    pc = metrics['pruned_cross'].mean(axis=-1)
    ratio = np.where(pw > 0, pc / np.maximum(pw, 1e-9), np.nan)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=ratio, mode='lines',
                             name='cross / within prune ratio'))
    fig.add_hline(y=1.0, line_dash='dot',
                  annotation_text='1 (proportional to init)')
    fig.update_layout(title='Cross/within prune ratio',
                      xaxis_title='step', yaxis_title='ratio',
                      width=900, height=420)
    fig.show()
    return fig


def plot_remaining_fractions(metrics):
    """Avg fraction of initial within/cross that are still active."""
    steps = metrics['steps']
    rw = metrics['frac_remaining_within'].mean(axis=-1)
    rc = metrics['frac_remaining_cross'].mean(axis=-1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=rw, mode='lines', name='within-task remaining',
                             line=dict(color=WITHIN_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=rc, mode='lines', name='cross-task remaining',
                             line=dict(color=CROSS_COLOR)))
    fig.update_layout(title='Fraction of initial input connections remaining',
                      xaxis_title='step',
                      yaxis_title='fraction remaining',
                      yaxis=dict(range=[0, 1.05]),
                      width=900, height=420)
    fig.show()
    return fig

In [10]:
plot_loss(deep_r_snaps, baseline_snaps)
plot_pruned_counts(metrics)
plot_pruned_ratio(metrics)
plot_remaining_fractions(metrics);